# C7 Currency-Level Exposure Budget — Phase 1
Fixed baseline diagnostic. Plan is frozen before implementation, implementation before outcomes. 2022–2026 is not a pristine unseen holdout. No Phase 2 or live changes.


In [ ]:
from pathlib import Path
import subprocess, sys, pandas as pd
BASELINE = Path('/content/daily_stop_baseline_trades.csv')
R2_ASSIGNMENT = Path('/content/c4_vol_change_phase1_trade_assignments_local.csv')
REPO = Path('/content/time-entry-portfolio-lab')
OUT = Path('/content')
IMPLEMENTATION_SHA = ''  # Set to frozen implementation SHA from Result; never use an unpinned branch.
SAVE_TO_DRIVE = False
DRIVE_DESTINATION = Path('/content/drive/MyDrive/c7_currency_exposure_phase1')
assert len(IMPLEMENTATION_SHA) == 40, 'Set the frozen implementation SHA before running'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/TR-KJ/time-entry-portfolio-lab.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout',IMPLEMENTATION_SHA],check=True)


In [ ]:
import os
run_env = dict(os.environ, OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1')
subprocess.run([sys.executable, str(REPO/'tests/test_c7_currency_exposure_phase1.py')],check=True,env=run_env)
subprocess.run([sys.executable, str(REPO/'src/research/c7_currency_exposure_phase1.py'),
 '--baseline',str(BASELINE),'--r2-assignment',str(R2_ASSIGNMENT),'--out',str(OUT),
 '--implementation-sha',IMPLEMENTATION_SHA],check=True,env=run_env)
subprocess.run([sys.executable, str(REPO/'tests/verify_c7_currency_exposure_phase1.py'),
 '--baseline',str(BASELINE),'--r2-assignment',str(R2_ASSIGNMENT),'--out',str(OUT)],check=True,env=run_env)
def table(name):
    return pd.read_csv(OUT/('c7_currency_exposure_phase1_'+name+'.csv'))


## 1. Baseline / Universe Audit

In [ ]:
display(table('trade_feature_summary')); display(table('validation'))

## 2. Open Position Reconstruction Audit

In [ ]:
display(table('independent_verification')); display(table('design_audit'))

## 3. Currency Exposure Examples

In [ ]:
display(table('manual_audit'))

## 4. NIC Distribution

In [ ]:
display(table('nic_distribution'))

## 5. Raw DIVERSIFYING / NEUTRAL / CONCENTRATING

In [ ]:
display(table('raw_sign_summary'))

## 6. Formal beta_NIC

In [ ]:
display(table('portfolio_summary').query("Variant == 'Primary'"))

## 7. Weekly cluster bootstrap 95%CI

In [ ]:
display(table('period_summary')[['Variant','Period','Beta','CILower','CIUpper','ValidBootstrap']])

## 8. Historical / Recent Stability

In [ ]:
display(table('period_summary').query("Variant == 'Primary'"))

## 9. Equal-unit Robustness

In [ ]:
display(table('equal_unit_robustness'))

## 10. Same-symbol-included Robustness

In [ ]:
display(table('same_symbol_robustness'))

## 11. Strategy / Symbol Diagnostic

In [ ]:
display(table('strategy_summary')); display(table('symbol_summary'))

## 12. Currency-Level / Simultaneous Diagnostic

In [ ]:
display(table('currency_summary')); display(table('simultaneous_diagnostic'))

## 13. Leave-One-Strategy-Out

In [ ]:
display(table('leave_one_strategy_out'))

## 14. Formal Gate A–H

In [ ]:
display(table('formal_gates'))

## 15. Final Verdict

In [ ]:
display(table('run_record')[['Verdict','PlanSHA','ImplementationSHA']])

## 16. Phase 2 eligibility / operational scope

In [ ]:
display(table('run_record')[['Phase2Eligible','Phase2Run','LiveChanged']])

## Optional Drive copy
Default OFF. Local CSVs include trade-level assignments and bootstrap draws. No Google Drive mounting or saving unless explicitly enabled.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    DRIVE_DESTINATION.mkdir(parents=True, exist_ok=True)
    for path in OUT.glob('c7_currency_exposure_phase1_*.csv'):
        shutil.copy2(path, DRIVE_DESTINATION/path.name)
